In [2]:
import pandas as pd
import numpy as np
df_raw = pd.read_csv(file_path, header=None)

df_data = df_raw.iloc[5:, :10].copy()

# Penamaan kolom awal
df_data.columns = [
    'Provinsi',
    'P3_Urban_Sem1', 'P3_Urban_Sem2', 'P3_Urban_Annual',
    'P3_Rural_Sem1', 'P3_Rural_Sem2', 'P3_Rural_Annual',
    'P3_Total_Sem1', 'P3_Total_Sem2', 'P3_Total_Annual'
]

# Clean string kolom Provinsi
df_data = df_data.dropna(subset=['Provinsi']).reset_index(drop=True)
df_data['Provinsi'] = df_data['Provinsi'].str.strip().str.title()

# Konversi kolom angka dari String/Koma ke Float Numeric
num_cols = ['P3_Urban_Sem1', 'P3_Urban_Sem2', 'P3_Rural_Sem1', 'P3_Rural_Sem2', 'P3_Total_Sem1', 'P3_Total_Sem2']
for col in num_cols:
    df_data[col] = pd.to_numeric(df_data[col].astype(str).str.replace('-', '').str.replace(',', '.'), errors='coerce')

# Hapus kolom Annual/Tahunan yang kosong (Feature Selection)
df_clean = df_data.drop(columns=['P3_Urban_Annual', 'P3_Rural_Annual', 'P3_Total_Annual']).copy()

# Penamaan ulang kolom yang rapi
df_clean.columns = [
    'Provinsi',
    'Kemiskinan_Perkotaan_Sem1',
    'Kemiskinan_Perkotaan_Sem2',
    'Kemiskinan_Perdesaan_Sem1',
    'Kemiskinan_Perdesaan_Sem2',
    'Kemiskinan_Total_Sem1',
    'Kemiskinan_Total_Sem2'
]


# Fitur Baru 1: Disparitas/Kesenjangan Desa vs Kota Semester 1
df_clean['Disparitas_Desa_Kota_Sem1'] = (df_clean['Kemiskinan_Perdesaan_Sem1'] - df_clean['Kemiskinan_Perkotaan_Sem1']).round(2)

# Fitur Baru 2: Disparitas/Kesenjangan Desa vs Kota Semester 2
df_clean['Disparitas_Desa_Kota_Sem2'] = (df_clean['Kemiskinan_Perdesaan_Sem2'] - df_clean['Kemiskinan_Perkotaan_Sem2']).round(2)

# Fitur Baru 3: Delta Perubahan Kemiskinan Total (Semester 2 - Semester 1)
df_clean['Perubahan_Kemiskinan_Sem1_Sem2'] = (df_clean['Kemiskinan_Total_Sem2'] - df_clean['Kemiskinan_Total_Sem1']).round(2)

# Fitur Baru 4: Status Tren Kemiskinan (Kategorikal)
def set_trend(val):
    if pd.isna(val): return 'Tidak Ada Data'
    if val < 0: return 'Penurunan Kemiskinan (Membaik)'
    elif val > 0: return 'Peningkatan Kemiskinan (Memburuk)'
    else: return 'Stagnan'

df_clean['Status_Tren_Kemiskinan'] = df_clean['Perubahan_Kemiskinan_Sem1_Sem2'].apply(set_trend)

# Fitur Baru 5: Kategori Tingkat Kemiskinan (Segmentasi)
def set_category(val):
    if pd.isna(val): return 'Tidak Ada Data'
    if val >= 15.0: return 'Sangat Tinggi (≥15%)'
    elif val >= 10.0: return 'Tinggi (10% - 14.99%)'
    elif val >= 6.0: return 'Sedang (6% - 9.99%)'
    else: return 'Rendah (<6%)'

df_clean['Kategori_Tingkat_Kemiskinan'] = df_clean['Kemiskinan_Total_Sem2'].apply(set_category)

df_clean.to_csv('Final_Clean_Dataset.csv', index=False)
df_clean.to_excel('Final_Clean_Dataset.xlsx', index=False)

print("--- EXEKUSI HARI KAMIS SELESAI ---")
print("Dataset Final berhasil disimpan!")
print(df_clean.head())

--- EXEKUSI HARI KAMIS SELESAI ---
Dataset Final berhasil disimpan!
         Provinsi  Kemiskinan_Perkotaan_Sem1  Kemiskinan_Perkotaan_Sem2  \
0            Aceh                       8.54                       8.15   
1  Sumatera Utara                       7.10                       7.16   
2  Sumatera Barat                       3.91                       3.75   
3            Riau                       5.75                       5.61   
4           Jambi                       9.52                       9.25   

   Kemiskinan_Perdesaan_Sem1  Kemiskinan_Perdesaan_Sem2  \
0                      14.44                      14.51   
1                       7.71                       7.35   
2                       6.93                       7.03   
3                       6.43                       6.76   
4                       6.01                       5.70   

   Kemiskinan_Total_Sem1  Kemiskinan_Total_Sem2  Disparitas_Desa_Kota_Sem1  \
0                  12.33                  12.22 